## 1. DevelopConfig — 파라미터 추가

In [ ]:
@dataclass
class DevelopConfig:
    S0: float = 100.0
    T: float = 1.0
    sigma: float = 4.0
    drift: float = 0.0
    dt: float = 0.005
    n_steps: int = 200
    gamma: float = 0.1
    A: float = 140.0
    k: float = 1.5
    q0: int = 0
    n_levels: int = 3          # ← 추가
    tick_size: float = 0.05    # ← 추가
    buckets: List[Tuple[float, float]] = field(default_factory=lambda: [
        (0.02, 1.0),
        (0.05, 1.5),
        (0.10, 2.5),
        (float('inf'), 3.5)
    ])

## 2. SimulationState — 필드 타입 변경 및 추가

In [ ]:
@dataclass
class SimulationState:
    step: int
    time: float
    mid_price: float
    inventory: int
    cash: float
    reservation_price: float
    spread: float
    bid: float
    ask: float
    bid_fill: int                 # ← bool → int (체결된 레벨 수)
    ask_fill: int                 # ← bool → int
    bid_prob: float = 0.0
    ask_prob: float = 0.0
    price_change_pct: float = 0.0
    multiplier: float = 1.0
    bid_fill_price: float = 0.0   # 총 bid 체결 금액
    ask_fill_price: float = 0.0   # 총 ask 체결 금액
    bid_levels_detail: str = ""   # ← 추가: 레벨별 체결 상세 (예: "1,1,0")
    ask_levels_detail: str = ""   # ← 추가

## 3. run_develop_simulation — 체결 로직 multi-level화

In [ ]:
def run_develop_simulation(config: DevelopConfig, mid_prices: np.ndarray = None) -> List[SimulationState]:
    if mid_prices is None:
        mid_prices = generate_mid_price_path(config)
    
    inventory = config.q0
    cash = 0.0
    history = []
    
    for step in range(config.n_steps):
        t = step * config.dt
        s = mid_prices[step]
        is_last_step = (step == config.n_steps - 1)
        s_next = mid_prices[step + 1] if not is_last_step else s
        
        r = compute_reservation_price(s, inventory, t, config)
        spread = compute_optimal_spread(t, config)
        bid, ask = compute_quotes(s, inventory, t, config)
        
        # ===== 변경: multi-level 호가 생성 =====
        bid_levels = [bid - i * config.tick_size for i in range(config.n_levels)]
        ask_levels = [ask + i * config.tick_size for i in range(config.n_levels)]
        
        # ===== 변경: 레벨별 체결 판정 =====
        bid_fills_raw = []
        ask_fills_raw = []
        bid_probs = []
        ask_probs = []
        
        for i in range(config.n_levels):
            delta_bid_i = s - bid_levels[i]
            delta_ask_i = ask_levels[i] - s
            
            prob_bid_i = compute_fill_probability_develop(
                delta_bid_i, s, s_next, True, config, is_last_step)
            prob_ask_i = compute_fill_probability_develop(
                delta_ask_i, s, s_next, False, config, is_last_step)
            
            bid_fills_raw.append(np.random.random() < prob_bid_i)
            ask_fills_raw.append(np.random.random() < prob_ask_i)
            bid_probs.append(prob_bid_i)
            ask_probs.append(prob_ask_i)
        
        # ===== 변경: sweep 보정 =====
        bid_fills = list(bid_fills_raw)
        ask_fills = list(ask_fills_raw)
        
        for i in range(config.n_levels - 1, 0, -1):
            if bid_fills[i]:
                bid_fills[i-1] = True
        for i in range(config.n_levels - 1, 0, -1):
            if ask_fills[i]:
                ask_fills[i-1] = True
        
        # ===== 변경: 레벨별 체결 가격 결정 =====
        total_bid_cost = 0.0
        total_ask_revenue = 0.0
        bid_fill_count = 0
        ask_fill_count = 0
        
        for i in range(config.n_levels):
            if bid_fills[i]:
                delta_bid_i = s - bid_levels[i]
                fill_price = s if delta_bid_i <= 0 else bid_levels[i]
                total_bid_cost += fill_price
                bid_fill_count += 1
            if ask_fills[i]:
                delta_ask_i = ask_levels[i] - s
                fill_price = s if delta_ask_i <= 0 else ask_levels[i]
                total_ask_revenue += fill_price
                ask_fill_count += 1
        
        price_change_pct = (s_next - s) / s * 100 if not is_last_step else 0.0
        multiplier = get_fill_multiplier(price_change_pct, config) if not is_last_step else 1.0
        
        bid_detail = ",".join(["1" if f else "0" for f in bid_fills])
        ask_detail = ",".join(["1" if f else "0" for f in ask_fills])
        
        state = SimulationState(
            step, t, s, inventory, cash, r, spread, bid, ask,
            bid_fill_count, ask_fill_count,
            bid_probs[0], ask_probs[0],
            price_change_pct, multiplier,
            total_bid_cost, total_ask_revenue,
            bid_detail, ask_detail
        )
        history.append(state)
        
        # ===== 변경: 체결 수량만큼 재고/현금 반영 =====
        inventory += bid_fill_count
        inventory -= ask_fill_count
        cash -= total_bid_cost
        cash += total_ask_revenue
    
    final_state = SimulationState(
        config.n_steps, config.T, mid_prices[-1], inventory, cash,
        mid_prices[-1], 0, mid_prices[-1], mid_prices[-1],
        0, 0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, "", ""
    )
    history.append(final_state)
    return history